# DataMate — Baseline Notebook (Zalo AI Challenge 2026)

**Bài toán:** Xây AI agent trả lời câu hỏi tiếng Việt đời thường trên dữ liệu bảng "bẩn" của hộ kinh doanh nhỏ (CSV / Excel nhiều sheet / SQLite), hoặc từ chối khi dữ liệu không đủ.

| Mục | Nội dung |
|---|---|
| Input | `questions.jsonl` (`id, business_id, question, as_of_date`) + gói dữ liệu của business (file + `README.md`) + **rulebook** và **lịch âm–dương** dùng chung |
| Output | `submission.jsonl`: `{"id", "type", "answer"}` với type ∈ `number, percent, string, date, boolean, list, table, unanswerable, abstain` |
| Metric | **Penalized Accuracy**: đúng **+1**, abstain **0**, sai **−1** (trả lời câu *unanswerable* = sai) |
| Ràng buộc | ≤ **15B tham số tổng** (tất cả model dùng khi inference), **offline** trong Docker, 2×RTX 5090, **≤ 6 giờ** cho toàn bộ private test, kết quả tái lập trong ±0.03 PA |

### Hệ quả quan trọng nhất của metric
Nếu agent đúng với xác suất `p` thì kỳ vọng điểm khi trả lời là `p − (1−p) = 2p − 1`, còn abstain là `0`.
→ **Chỉ trả lời khi `p > 0.5`.** Một hệ thống đúng 60% nhưng trả lời mọi câu chỉ được PA = 0.20; nếu nó lọc được để trả lời 50% câu với độ chính xác 90% thì PA = 0.40. **Calibration (ước lượng độ tự tin) quan trọng ngang độ chính xác.**

### Kiến trúc baseline
```
câu hỏi + schema profile + README + rulebook + few-shot (truy hồi từ train)
        │
        ▼
LLM ≤15B (vLLM)  ── sinh K chương trình Python/DuckDB (self-consistency)
        │
        ▼
Sandbox thực thi (subprocess, timeout) ── lỗi? → 1 vòng "repair" với traceback
        │
        ▼
Chuẩn hoá đáp án → bỏ phiếu đa số → confidence = tỉ lệ phiếu
        │
        ▼
confidence ≥ τ(type) ? trả lời : abstain      (τ hiệu chỉnh trên train để max PA)
```

### Mục lục
0. Cài đặt & cấu hình
1. Khám phá cấu trúc dữ liệu
2. Loader chịu được dữ liệu bẩn (header lệch, nhiều sheet, số kiểu VN, dòng tổng)
3. Schema profiling — "ống kính" cho LLM
4. Bộ chấm điểm local (tái hiện đúng luật của BTC)
5. Sandbox thực thi code
6. LLM backend (vLLM offline / server OpenAI-compatible)
7. Prompt + truy hồi few-shot
8. Agent: sinh → chạy → sửa → bỏ phiếu
9. Chạy trên train, hiệu chỉnh ngưỡng abstain
10. Chạy test & ghi `submission.jsonl` (an toàn thời gian)
11. Lộ trình để đạt điểm cao

## 0a. Chạy trên Kaggle (dev / public test)

Kaggle miễn phí cho **2×T4 (16 GB mỗi card)** hoặc P100, phiên tối đa 12h, quota khoảng 30 GPU-giờ/tuần. Cần lưu ý:
- **Chọn Accelerator = GPU T4 x2** (P100 compute 6.0 không chạy được vLLM).
- T4 **không hỗ trợ bf16** → dùng `dtype="half"`; Qwen3-14B bản bf16 (~30 GB) không vừa 2×16 GB khi còn KV cache → dùng bản **AWQ 4-bit** (`Qwen/Qwen3-14B-AWQ`, ~10 GB) với `tensor_parallel_size=2`.
- Bật **Internet** trong Settings để `pip install vllm` và tải model (chỉ lúc dev; Docker nộp cho BTC phải offline).
- Add dataset DataMate vào notebook (Add Input) → dữ liệu nằm ở `/kaggle/input/<tên-dataset>/`.

In [ ]:
# Chỉ chạy trên Kaggle (bỏ qua nếu chạy ở máy khác)
import os, subprocess
if os.path.exists("/kaggle"):
    subprocess.run("pip install -q 'vllm>=0.10' duckdb openpyxl rapidfuzz", shell=True)
    print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True,
                         capture_output=True, text=True).stdout)

## 0. Cài đặt & cấu hình

Model mặc định: **Qwen3-14B** (14.8B tham số, Apache-2.0) — vừa giới hạn 15B nhưng **không còn chỗ cho embedding/reranker** (luật tính *tổng* tham số mọi model dùng khi inference). Vì vậy few-shot retrieval ở đây dùng TF-IDF (không tham số học sâu). Phương án khác: `Qwen2.5-Coder-14B-Instruct` (14.7B), hoặc một model ~8B để chừa chỗ cho model phụ. Hãy kiểm tra các model ≤15B mới hơn được công bố **trước 26/10/2026**.

> Trên Blackwell (RTX 5090) cần CUDA ≥ 12.8 và bản PyTorch/vLLM tương ứng — dùng base image theo Docker guideline của BTC.

In [ ]:
# !pip install -q "vllm>=0.10" duckdb openpyxl pandas scikit-learn rapidfuzz
import os, sys, re, json, time, math, pickle, random, hashlib, unicodedata, subprocess, tempfile, textwrap
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
try:
    display
except NameError:
    display = print

import numpy as np
import pandas as pd

# ---------------- CONFIG ----------------
ON_KAGGLE = os.path.exists("/kaggle/input")
if ON_KAGGLE:
    # Tự dò dataset: train = thư mục có answers.jsonl, test = có questions.jsonl nhưng không có answers.jsonl
    _KIN = Path("/kaggle/input")
    _qdirs = sorted({p.parent for p in _KIN.rglob("questions.jsonl")}, key=lambda d: len(d.parts))
    for d in _qdirs: print("found:", d, "| answers.jsonl:", (d / "answers.jsonl").exists())
    _train = next((d for d in _qdirs if (d / "answers.jsonl").exists()), None)
    _tests = [d for d in _qdirs if not (d / "answers.jsonl").exists()]
    _test = next((d for d in _tests if "test" in str(d).lower()), _tests[0] if _tests else None)
    assert _train is not None, "Không thấy train (answers.jsonl) trong /kaggle/input — đã Add Input dataset chưa?"
    assert _test is not None, "Không thấy test (questions.jsonl không kèm answers.jsonl) trong /kaggle/input"
    _defaults = dict(TRAIN_ROOT=str(_train), TEST_ROOT=str(_test),
                     RESULT_PATH="/kaggle/working/result/submission.jsonl", WORK_DIR="/kaggle/working/work",
                     MODEL_NAME="Qwen/Qwen3-14B-AWQ")
else:
    # Tự động nhận diện thư mục DataMate (chạy từ trong DataMate/ hoặc từ thư mục gốc ZALOAI/)
    _BASE_DIR = Path(".").resolve() if (Path("./train").exists() and Path("./public_test").exists()) else (
                Path("./DataMate").resolve() if Path("./DataMate/train").exists() else Path(".").resolve())
    if not (_BASE_DIR / "train" / "questions.jsonl").exists():
        import urllib.request, urllib.parse
        _raw_base = os.environ.get("DATAMATE_RAW_URL", "https://raw.githubusercontent.com/vdhwy/ZALOAI/datamate/DataMate")
        try:
            print(f"[Dataset] Đang tải DataMate từ {_raw_base} ...")
            with urllib.request.urlopen(f"{_raw_base}/manifest.json", timeout=30) as _r:
                _manifest = json.loads(_r.read().decode("utf-8"))
            def _dl_one(_item):
                _dst = _BASE_DIR / _item["path"]
                if _dst.exists() and _dst.stat().st_size == _item["size"]: return True
                _dst.parent.mkdir(parents=True, exist_ok=True)
                _url = f"{_raw_base}/" + "/".join(urllib.parse.quote(p) for p in _item["path"].split("/"))
                for _ in range(4):
                    try:
                        with urllib.request.urlopen(_url, timeout=60) as _resp:
                            _data = _resp.read()
                        if len(_data) == _item["size"]:
                            _dst.write_bytes(_data)
                            return True
                    except Exception:
                        time.sleep(1)
                return False
            with ThreadPoolExecutor(max_workers=16) as _ex:
                _ok = sum(1 for _res in _ex.map(_dl_one, _manifest) if _res)
            print(f"[Dataset] Đã tải xong {_ok}/{len(_manifest)} file vào {_BASE_DIR}")
        except Exception as _e:
            print(f"[Dataset Warning] Không tải được tự động: {_e}")
    _defaults = dict(TRAIN_ROOT=str(_BASE_DIR / "train"),
                     TEST_ROOT=str(_BASE_DIR / "public_test") if (_BASE_DIR / "public_test").exists() else "/data",
                     RESULT_PATH=str(_BASE_DIR / "result" / "submission.jsonl"),
                     WORK_DIR="/tmp/datamate_work",
                     MODEL_NAME="Qwen/Qwen3-14B")
TRAIN_ROOT  = Path(os.environ.get("TRAIN_ROOT", _defaults["TRAIN_ROOT"]))
TEST_ROOT   = Path(os.environ.get("TEST_ROOT",  _defaults["TEST_ROOT"]))
RESULT_PATH = Path(os.environ.get("RESULT_PATH", _defaults["RESULT_PATH"]))
LOG_DIR     = RESULT_PATH.parent / "logs"                                   # BTC yêu cầu log từng câu ở /result/logs/
WORK_DIR    = Path(os.environ.get("WORK_DIR", _defaults["WORK_DIR"]))       # /data là read-only → ghi tạm ở đây

MODEL_NAME  = os.environ.get("MODEL_NAME", _defaults["MODEL_NAME"])
BACKEND     = os.environ.get("BACKEND", "vllm")      # "vllm" | "openai" (server local, chỉ để dev) | "mock"
OPENAI_BASE = os.environ.get("OPENAI_BASE", "http://localhost:8000/v1")

N_SAMPLES       = int(os.environ.get("N_SAMPLES", 3))        # K chương trình / câu (self-consistency)
TEMPERATURE     = 0.7
MAX_NEW_TOKENS  = int(os.environ.get("MAX_NEW_TOKENS", 1024))
REPAIR_ROUNDS   = 1
EXEC_TIMEOUT    = 30       # giây / chương trình
EXEC_WORKERS    = max(4, (os.cpu_count() or 8) - 2)
CHUNK_SIZE      = 64       # số câu xử lý mỗi đợt (ghi file sau mỗi đợt)
TIME_BUDGET_S   = float(os.environ.get("TIME_BUDGET_H", 11 if ON_KAGGLE else 5.3)) * 3600   # Kaggle phiên 12h; Docker BTC 6h
# Hiệu chỉnh (train) khi có tập train (hoặc trên Kaggle); Docker BTC (2×RTX 5090, 6h) có thể đặt SKIP_CALIB=1 để đọc artifacts.json đóng gói sẵn
SKIP_CALIB      = os.environ.get("SKIP_CALIB", "0" if (ON_KAGGLE or TRAIN_ROOT.exists()) else "1") == "1"
CALIB_BUDGET_S  = float(os.environ.get("CALIB_BUDGET_H", 5)) * 3600   # Kaggle: ≤5h hiệu chỉnh, còn lại chạy public test
ARTIFACTS_PATH  = Path(os.environ.get("ARTIFACTS_PATH", "/kaggle/working/artifacts.json" if ON_KAGGLE else str(RESULT_PATH.parent.parent / "artifacts.json")))
SEED            = 2026
DEFAULT_TAU     = 0.6      # ngưỡng abstain mặc định nếu chưa hiệu chỉnh

random.seed(SEED); np.random.seed(SEED)
WORK_DIR.mkdir(parents=True, exist_ok=True)
T_START = time.time()
print("Train:", TRAIN_ROOT, "| Test:", TEST_ROOT, "| Model:", MODEL_NAME, "| Backend:", BACKEND)

## 1. Khám phá cấu trúc dữ liệu

Cấu trúc mỗi split theo đề:
```
<split>/
├── data/<business_id>/   *.csv | *.xlsx | *.sqlite + README.md
├── rulebook.md           quy ước mặc định (KHÁC NHAU giữa các split → luôn đọc rulebook của split đang chạy)
├── calendar.csv          lịch âm–dương (Tết)
├── questions.jsonl       {id, business_id, question, as_of_date}
└── answers.jsonl         chỉ train: {id, type, answer, alias}
```
`alias` = các dạng tên khác được chấp nhận (cách viết khác trong dữ liệu + mã hàng). Câu unanswerable có `answer: null`.
Hàm `discover()` dùng đường dẫn chuẩn này, nếu không thấy thì tự dò.

In [ ]:
def tree(root, max_entries=40):
    root = Path(root)
    if not root.exists():
        print(f"[!] {root} không tồn tại"); return
    n = 0
    for p in sorted(root.rglob("*")):
        depth = len(p.relative_to(root).parts)
        if depth <= 3:
            print("  " * (depth - 1) + ("📁 " if p.is_dir() else "📄 ") + p.name)
            n += 1
            if n >= max_entries:
                print("  ..."); break

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

def discover(root):
    root = Path(root)
    std = {"questions": root / "questions.jsonl", "labels": root / "answers.jsonl",
           "rulebook": root / "rulebook.md", "calendar": root / "calendar.csv"}
    if std["questions"].exists() and (root / "data").is_dir():
        info = {"root": root, **{k: (v if v.exists() else None) for k, v in std.items()},
                "biz_dirs": {p.name: p for p in sorted((root / "data").iterdir()) if p.is_dir()}}
        return info
    # fallback: tự dò nếu cấu trúc khác
    info = {"root": root, "questions": None, "labels": None, "biz_dirs": {}, "rulebook": None, "calendar": None}
    jsonls = sorted(root.rglob("*.jsonl"))
    for p in jsonls:
        rows = read_jsonl(p)[:3]
        if not rows: continue
        keys = set(rows[0])
        if {"question", "business_id"} <= keys and info["questions"] is None:
            info["questions"] = p
        elif {"id", "answer"} <= keys and "question" not in keys:
            info["labels"] = p
    for p in root.rglob("*"):
        if p.is_dir() and re.match(r"^biz[_-]?\w+$", p.name):
            info["biz_dirs"][p.name] = p
        low = p.name.lower()
        if p.is_file() and info["rulebook"] is None and ("rule" in low) and p.suffix in {".md", ".txt", ".json", ".yaml", ".yml"}:
            info["rulebook"] = p
        if p.is_file() and info["calendar"] is None and any(k in low for k in ("calendar", "lunar", "am_lich", "amlich")):
            info["calendar"] = p
    return info

for name, root in [("TRAIN", TRAIN_ROOT), ("TEST", TEST_ROOT)]:
    print(f"===== {name}: {root}")
    tree(root, 30)
    if Path(root).exists():
        d = discover(root)
        print({k: (str(v) if not isinstance(v, dict) else f"{len(v)} businesses") for k, v in d.items()})

In [ ]:
def load_split(root):
    info = discover(root)
    if info["questions"] is None:
        return info, pd.DataFrame()
    q = pd.DataFrame(read_jsonl(info["questions"]))
    if info["labels"] is not None:
        lab = pd.DataFrame(read_jsonl(info["labels"]))
        lab = lab.rename(columns={"type": "gold_type", "answer": "gold_answer", "alias": "gold_alias"})
        q = q.merge(lab, on="id", how="left")
    else:  # nhãn nằm ngay trong questions.jsonl
        q = q.rename(columns={"type": "gold_type", "answer": "gold_answer", "alias": "gold_alias"})
    if "gold_type" in q:
        if "gold_alias" not in q: q["gold_alias"] = None
        q["gold"] = [gold_with_alias(t, a, al) for t, a, al in zip(q.gold_type, q.gold_answer, q.gold_alias)]
    return info, q

def gold_with_alias(t, ans, alias):
    # gộp alias vào đáp án chuẩn để bộ chấm local chấp nhận mọi dạng tên hợp lệ
    if not isinstance(alias, list) or not alias:
        return ans
    if t == "string":
        return [ans] + [a for a in alias if a is not None]
    if t == "list" and isinstance(ans, list):
        if len(alias) == len(ans) and all(isinstance(a, list) for a in alias):   # alias theo từng phần tử
            return [[x] + al if isinstance(x, str) else x for x, al in zip(ans, alias)]
    return ans

train_info, train_df = load_split(TRAIN_ROOT) if TRAIN_ROOT.exists() else (None, pd.DataFrame())
test_info,  test_df  = load_split(TEST_ROOT)  if TEST_ROOT.exists()  else (None, pd.DataFrame())
print("train:", train_df.shape, "| test:", test_df.shape)
if len(train_df):
    display(train_df.head())
    if "gold_type" in train_df:
        print("\nPhân bố loại đáp án (train):")
        print(train_df["gold_type"].value_counts(normalize=True).round(3))
    print("\nSố câu / business:", train_df.groupby("business_id").size().describe().round(1).to_dict())

**Những gì cần nhìn trong EDA** (ảnh hưởng trực tiếp tới điểm):
- Tỉ lệ `unanswerable` trong train → prior cho bộ phân loại "từ chối".
- Phân bố `type` → loại nào nhiều thì tối ưu trước (thường `number` và `string`).
- Đọc tay ~30 câu: tiếng lóng/viết tắt (*"sp bán dc nhiều nhất tuần rồi"*), biểu thức thời gian tương đối, các khái niệm nghiệp vụ (giá vốn, công nợ quá hạn, đơn hợp lệ).

## 2. Loader chịu được dữ liệu bẩn

Dữ liệu hộ kinh doanh thật thường có: tiêu đề/tên cửa hàng ở vài dòng đầu (header lệch), ô gộp, nhiều sheet, số dạng `1.250.000đ`, ngày dạng `15/08/2026`, dòng **Tổng cộng** cuối bảng (gây đếm trùng). Loader dưới đây:
1. Đọc mọi sheet / mọi bảng SQLite thành `DataFrame`.
2. **Dò dòng header** (dòng có nhiều ô chữ nhất trong 15 dòng đầu).
3. **Ép kiểu** số/ngày kiểu Việt Nam khi ≥ 95% giá trị parse được.
4. **Loại dòng tổng** ở cuối bảng.
5. Đặt tên bảng an toàn cho SQL (`file__sheet`), cache bằng pickle.

In [ ]:
def strip_accents(s: str) -> str:
    s = unicodedata.normalize("NFD", str(s)).replace("đ", "d").replace("Đ", "D")
    return "".join(c for c in s if unicodedata.category(c) != "Mn")

def safe_ident(s: str) -> str:
    s = re.sub(r"[^0-9a-zA-Z]+", "_", strip_accents(s).lower()).strip("_")
    return ("t_" + s) if (not s or s[0].isdigit()) else s

def make_unique(cols):
    seen, out = Counter(), []
    for c in cols:
        c = str(c).strip() if c is not None and str(c) != "nan" else "col"
        seen[c] += 1
        out.append(c if seen[c] == 1 else f"{c}_{seen[c]}")
    return out

def detect_header_row(raw: pd.DataFrame, max_scan=15) -> int:
    best, best_score = 0, -1
    ncol = raw.shape[1]
    for i in range(min(max_scan, len(raw))):
        row = raw.iloc[i]
        nonnull = row.notna().sum()
        n_str = sum(isinstance(v, str) and not re.fullmatch(r"[\d.,\s%-]+", v) for v in row)
        score = n_str + 0.5 * nonnull
        if nonnull >= max(2, 0.5 * ncol) and score > best_score:
            best, best_score = i, score
    return best

_NUM_RE = re.compile(r"^[+-]?[\d.,\s]+(đ|d|vnd|vnđ|%)?$", re.I)
def parse_vn_number(v):
    if v is None or (isinstance(v, float) and math.isnan(v)): return np.nan
    if isinstance(v, (int, float, np.integer, np.floating)): return float(v)
    s = str(v).strip().lower()
    if not s or not _NUM_RE.match(s): return np.nan
    s = re.sub(r"(đ|vnđ|vnd|d|%)$", "", s).replace(" ", "")
    if "," in s and "." in s:                       # 1.250.000,5  hoặc 1,250,000.5
        s = s.replace(".", "").replace(",", ".") if s.rfind(",") > s.rfind(".") else s.replace(",", "")
    elif s.count(".") > 1 or re.fullmatch(r"\d{1,3}(\.\d{3})+", s):
        s = s.replace(".", "")                       # 1.250.000 (kiểu VN)
    elif s.count(",") > 1 or re.fullmatch(r"\d{1,3}(,\d{3})+", s):
        s = s.replace(",", "")                       # 1,250,000
    else:
        s = s.replace(",", ".")                      # 12,5 → 12.5
    try: return float(s)
    except ValueError: return np.nan

def coerce_types(df: pd.DataFrame, min_ok=0.95) -> pd.DataFrame:
    for c in df.columns:
        col = df[c]
        if col.dtype != object: continue
        nn = col.dropna()
        if len(nn) == 0: continue
        nums = nn.map(parse_vn_number)
        if nums.notna().mean() >= min_ok:
            df[c] = col.map(parse_vn_number); continue
        if nn.astype(str).str.contains(r"\d{1,4}[-/]\d{1,2}[-/]\d{1,4}").mean() >= min_ok:
            iso = nn.astype(str).str.match(r"^\d{4}-").mean() > 0.5
            dt_ = pd.to_datetime(col, errors="coerce", dayfirst=not iso, format="mixed")
            if dt_.notna().sum() >= min_ok * len(nn):
                df[c] = dt_
    return df

_TOTAL_RE = re.compile(r"^\s*(tổng|tong|total|cộng|cong)\b", re.I)
def drop_total_rows(df: pd.DataFrame, tail=3) -> pd.DataFrame:
    if len(df) < 4: return df
    idx = []
    for i in df.index[-tail:]:
        first_txt = next((v for v in df.loc[i] if isinstance(v, str) and v.strip()), None)
        if first_txt and _TOTAL_RE.match(first_txt): idx.append(i)
    return df.drop(index=idx)

def tidy(raw: pd.DataFrame) -> pd.DataFrame:
    raw = raw.dropna(how="all").dropna(axis=1, how="all")
    if raw.empty: return raw
    h = detect_header_row(raw)
    df = raw.iloc[h + 1:].copy()
    df.columns = make_unique(raw.iloc[h].tolist())
    df = df.reset_index(drop=True)
    df = drop_total_rows(df).reset_index(drop=True)
    return coerce_types(df)

def read_csv_robust(p):
    for enc in ("utf-8-sig", "utf-8", "cp1258", "latin-1"):
        try:
            return pd.read_csv(p, header=None, sep=None, engine="python", encoding=enc, dtype=object)
        except Exception:
            continue
    raise RuntimeError(f"Không đọc được {p}")

def load_business(biz_dir: Path):
    tables, sources = {}, {}
    for p in sorted(biz_dir.rglob("*")):
        if not p.is_file(): continue
        suf, stem = p.suffix.lower(), safe_ident(p.stem)
        try:
            if suf == ".csv":
                tables[stem] = tidy(read_csv_robust(p)); sources[stem] = p.name
            elif suf in (".xlsx", ".xlsm", ".xls"):
                for sh, raw in pd.read_excel(p, sheet_name=None, header=None, dtype=object).items():
                    k = f"{stem}__{safe_ident(sh)}"
                    t = tidy(raw)
                    if not t.empty: tables[k] = t; sources[k] = f"{p.name} › sheet '{sh}'"
            elif suf in (".sqlite", ".db", ".sqlite3"):
                import sqlite3
                with sqlite3.connect(f"file:{p}?mode=ro", uri=True) as con:
                    names = [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type IN ('table','view')")]
                    for n in names:
                        k = safe_ident(n) if safe_ident(n) not in tables else f"{stem}__{safe_ident(n)}"
                        tables[k] = coerce_types(pd.read_sql_query(f'SELECT * FROM "{n}"', con)); sources[k] = f"{p.name} › table '{n}'"
        except Exception as e:
            print(f"[warn] {p}: {e}")
    readme = next((x.read_text(encoding="utf-8", errors="ignore") for x in biz_dir.rglob("*") if x.name.lower() == "readme.md"), "")
    return {"tables": tables, "sources": sources, "readme": readme}

_BIZ_CACHE = {}
def get_business(info, biz_id):
    if biz_id in _BIZ_CACHE: return _BIZ_CACHE[biz_id]
    pkl = WORK_DIR / "cache" / f"{safe_ident(str(info['root']))}__{biz_id}.pkl"
    if pkl.exists():
        b = pickle.load(open(pkl, "rb"))
    else:
        b = load_business(info["biz_dirs"][biz_id])
        pkl.parent.mkdir(parents=True, exist_ok=True)
        pickle.dump(b, open(pkl, "wb"))
    b["pkl"] = str(pkl)
    _BIZ_CACHE[biz_id] = b
    return b

In [ ]:
# Rulebook & lịch âm–dương (dùng chung cho cả split)
def load_shared(info):
    rulebook = info["rulebook"].read_text(encoding="utf-8", errors="ignore") if info and info["rulebook"] else ""
    cal = None
    if info and info["calendar"]:
        p = info["calendar"]
        cal = tidy(read_csv_robust(p)) if p.suffix.lower() == ".csv" else (
              tidy(pd.read_excel(p, header=None, dtype=object)) if p.suffix.lower() in (".xlsx", ".xls") else None)
        if p.suffix.lower() == ".json":
            cal = pd.DataFrame(json.load(open(p, encoding="utf-8")))
    cal_pkl = WORK_DIR / f"calendar_{safe_ident(str(info['root'])) if info else 'none'}.pkl"
    pickle.dump(cal, open(cal_pkl, "wb"))
    return rulebook, cal, str(cal_pkl)

if train_info:
    TRAIN_RULEBOOK, TRAIN_CAL, TRAIN_CAL_PKL = load_shared(train_info)
    print("Rulebook:", len(TRAIN_RULEBOOK), "ký tự")
    print(TRAIN_RULEBOOK[:1500])
    if TRAIN_CAL is not None: display(TRAIN_CAL.head())

> ⚠️ **Đọc kỹ rulebook bằng mắt.** Nó định nghĩa những thứ quyết định đúng/sai: thế nào là *đơn hợp lệ* (ranking chỉ tính "valid orders"), tiêu chí xếp hạng mặc định, cách tính giá vốn/lãi gộp, công nợ quá hạn, tuần bắt đầu thứ mấy... Ở baseline, rulebook được nhét thẳng vào prompt; bản mạnh nên **chuyển nó thành hàm Python** (xem mục 11).

## 3. Schema profiling — "ống kính" cho LLM

LLM không thể đọc cả bảng, nên ta mô tả mỗi bảng gọn: nguồn file/sheet, số dòng, từng cột với kiểu, số giá trị khác nhau, **mẫu giá trị** (rất quan trọng để LLM viết đúng `WHERE ten_sp = ...`), min/max cho số và ngày. Với cột văn bản ít giá trị (danh mục, trạng thái đơn) ta liệt kê đầy đủ để LLM biết chính xác chuỗi cần lọc (`'Đã hủy'` vs `'huy'`...).

In [ ]:
def profile_table(name, df, src, max_cat=25, max_samples=6):
    lines = [f"### bảng `{name}`  (nguồn: {src}; {len(df)} dòng)"]
    for c in df.columns:
        s = df[c]; nn = s.dropna(); nu = nn.nunique()
        if pd.api.types.is_datetime64_any_dtype(s):
            desc = f"datetime, {nn.min().date() if len(nn) else '-'} → {nn.max().date() if len(nn) else '-'}"
        elif pd.api.types.is_numeric_dtype(s):
            desc = f"số, min={nn.min() if len(nn) else '-'}, max={nn.max() if len(nn) else '-'}, unique={nu}"
        else:
            vals = nn.astype(str).value_counts()
            if nu <= max_cat:
                desc = f"text, {nu} giá trị: " + ", ".join(repr(v) for v in vals.index[:max_cat])
            else:
                desc = f"text, unique={nu}, ví dụ: " + ", ".join(repr(v[:40]) for v in vals.index[:max_samples])
        null_pct = 100 * (1 - len(nn) / max(1, len(s)))
        lines.append(f"- `{c}`: {desc}" + (f" (null {null_pct:.0f}%)" if null_pct >= 5 else ""))
    return "\n".join(lines)

def business_profile(b, max_chars=14000):
    parts = [profile_table(k, v, b["sources"].get(k, "?")) for k, v in b["tables"].items()]
    txt = "\n\n".join(parts)
    return txt if len(txt) <= max_chars else txt[:max_chars] + "\n...(cắt bớt)"

if len(train_df):
    b0 = get_business(train_info, train_df.business_id.iloc[0])
    print(b0["readme"][:800]); print()
    print(business_profile(b0)[:4000])

## 4. Bộ chấm điểm local — tái hiện đúng luật BTC

Đây là phần **phải làm đúng trước tiên**: mọi quyết định (prompt, model, ngưỡng) đều dựa vào điểm local. Luật khớp theo từng type:

| type | luật khớp |
|---|---|
| `number` | khớp chính xác; gold là số nguyên hoặc làm tròn 2 chữ số |
| `percent` | khớp sau khi làm tròn 2 chữ số, đúng dấu (`-12.45`) |
| `string` | NFC → lowercase → ký tự không phải chữ/số/khoảng trắng thành space → gộp khoảng trắng; **giữ dấu**. Chấp nhận mọi cách viết tên xuất hiện trong dữ liệu **và cả mã** (vd. `sp0042`) |
| `date` | `yyyy-mm-dd`, câu hỏi về tháng: `yyyy-mm` |
| `boolean` | `true/false` |
| `list` | có thứ tự, so từng vị trí theo luật string/number |
| `table` | cột khớp theo tên chuẩn hoá (không được thì theo vị trí); dòng so như **tập hợp** |
| `unanswerable` | đúng khi và chỉ khi gold là unanswerable |
| `abstain` | 0 điểm |

`PA = mean(s_i)` với `s_i ∈ {+1, 0, −1}`.

In [ ]:
def norm_str(s) -> str:
    s = unicodedata.normalize("NFC", str(s)).lower()
    s = "".join(ch if (ch.isalnum() or ch.isspace()) else " " for ch in s)
    return re.sub(r"\s+", " ", s).strip()

def to_float(x):
    try:
        if isinstance(x, bool): return None
        return float(str(x).replace(",", "")) if isinstance(x, str) else float(x)
    except (TypeError, ValueError):
        return None

def num_eq(a, b):
    fa, fb = to_float(a), to_float(b)
    return fa is not None and fb is not None and round(fa, 2) == round(fb, 2)

def str_eq(a, gold_forms):
    forms = gold_forms if isinstance(gold_forms, list) else [gold_forms]
    return norm_str(a) in {norm_str(g) for g in forms}

def cell_eq(a, g):
    # gold cell có thể là số, chuỗi, hoặc list các dạng chấp nhận của một tên
    if isinstance(g, list): return str_eq(a, g)
    if to_float(g) is not None and not isinstance(g, str): return num_eq(a, g)
    return num_eq(a, g) if (to_float(a) is not None and to_float(g) is not None) else str_eq(a, g)

def table_eq(a, g):
    try:
        ac, gc = [norm_str(c) for c in a["columns"]], [norm_str(c) for c in g["columns"]]
        if len(ac) != len(gc) or len(a["rows"]) != len(g["rows"]): return False
        perm = [ac.index(c) for c in gc] if set(ac) == set(gc) else list(range(len(gc)))
        arows = [[r[i] for i in perm] for r in a["rows"]]
        unused = list(range(len(arows)))
        for gr in g["rows"]:
            hit = next((j for j in unused if all(cell_eq(x, y) for x, y in zip(arows[j], gr))), None)
            if hit is None: return False
            unused.remove(hit)
        return True
    except Exception:
        return False

def match(pred_type, pred_ans, gold_type, gold_ans) -> bool:
    if pred_type != gold_type: return False
    t = gold_type
    if t == "unanswerable": return True
    if t in ("number", "percent"): return num_eq(pred_ans, gold_ans)
    if t == "string": return str_eq(pred_ans, gold_ans)
    if t == "date": return str(pred_ans).strip() == str(gold_ans).strip()
    if t == "boolean": return str(pred_ans).lower() == str(gold_ans).lower()
    if t == "list":
        if not isinstance(pred_ans, list) or len(pred_ans) != len(gold_ans): return False
        return all(cell_eq(a, g) for a, g in zip(pred_ans, gold_ans))
    if t == "table": return table_eq(pred_ans, gold_ans)
    return False

def score_one(pred_type, pred_ans, gold_type, gold_ans) -> int:
    if pred_type in (None, "abstain"): return 0
    return 1 if match(pred_type, pred_ans, gold_type, gold_ans) else -1

def penalized_accuracy(preds: dict, gold_df: pd.DataFrame):
    s = [score_one(*preds.get(r.id, ("abstain", None)), r.gold_type, r.gold) for r in gold_df.itertuples()]
    s = np.array(s)
    return {"PA": s.mean(), "answered": (s != 0).mean(), "precision": (s == 1).sum() / max(1, (s != 0).sum())}

# unit tests nhanh
assert score_one("string", "Sting  dâu-330ml", "string", ["Sting dâu 330ml", "SP0042"]) == 1
assert score_one("string", "sp0042", "string", ["Sting dâu 330ml", "SP0042"]) == 1
assert score_one("string", "Sting dau 330ml", "string", ["Sting dâu 330ml"]) == -1   # mất dấu = sai
assert score_one("percent", -12.449, "percent", -12.45) == 1
assert score_one("number", 5, "unanswerable", None) == -1
assert score_one("abstain", None, "number", 5) == 0
assert score_one("table", {"columns": ["SL", "Tên"], "rows": [[3, "b"], [5, "a"]]}, "table",
                 {"columns": ["tên", "sl"], "rows": [["a", 5], ["b", 3]]}) == 1
print("scorer OK")

> 💡 **Mẹo từ luật khớp `string`:** đáp án đúng nếu trùng *bất kỳ* cách viết nào của tên trong dữ liệu **hoặc mã** của nó. Vì vậy agent nên trả về **chuỗi lấy nguyên văn từ dữ liệu** (hoặc mã sản phẩm) thay vì để LLM tự "viết lại" tên — tránh mất dấu, sai chính tả. Với `number`, luôn làm tròn 2 chữ số; tiền VND thường là số nguyên.

## 5. Sandbox thực thi code

Mỗi chương trình do LLM sinh ra chạy trong **subprocess riêng** có timeout (tránh treo/vòng lặp vô hạn làm hỏng ngân sách 6 giờ). Môi trường được nạp sẵn:
- `tables`: dict `tên_bảng → DataFrame` (đã làm sạch), `sql(q)`: chạy DuckDB SQL trên các bảng đó.
- `AS_OF`: `pd.Timestamp` "hôm nay"; `calendar`: bảng lịch âm–dương.
- Helper: `norm(s)` (chuẩn hoá giống bộ chấm), `find_name(text, series)` (fuzzy match tên sản phẩm/khách theo câu hỏi).
- Chương trình phải gán `ANSWER = {"type": ..., "answer": ...}`.

In [ ]:
RUNNER_SRC = r"""
import sys, json, pickle, re, unicodedata, math, datetime as dt, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
try:
    import duckdb
except ImportError:
    duckdb = None

biz = pickle.load(open(sys.argv[1], "rb"))
tables = biz["tables"]
calendar = pickle.load(open(sys.argv[2], "rb"))
AS_OF = pd.Timestamp(sys.argv[3])
TODAY = AS_OF

_con = duckdb.connect() if duckdb else None
if _con is not None:
    for _k, _v in tables.items():
        _con.register(_k, _v)
    if calendar is not None:
        _con.register("calendar", calendar)
def sql(q):
    return _con.execute(q).df()

def norm(s):
    s = unicodedata.normalize("NFC", str(s)).lower()
    s = "".join(ch if (ch.isalnum() or ch.isspace()) else " " for ch in s)
    return re.sub(r"\s+", " ", s).strip()

def find_name(text, series, limit=5):
    from rapidfuzz import process, fuzz
    choices = pd.Series(series).dropna().astype(str).unique().tolist()
    return process.extract(norm(text), choices, processor=norm, scorer=fuzz.WRatio, limit=limit)

ANSWER = None
exec(open(sys.argv[4], encoding="utf-8").read(), globals())

def _py(x):
    if isinstance(x, (np.integer,)): return int(x)
    if isinstance(x, (np.floating,)): return float(x)
    if isinstance(x, (np.bool_,)): return bool(x)
    if isinstance(x, (pd.Timestamp, dt.date, dt.datetime)): return x.strftime("%Y-%m-%d")
    if isinstance(x, (list, tuple, np.ndarray, pd.Series)): return [_py(v) for v in list(x)]
    if isinstance(x, dict): return {k: _py(v) for k, v in x.items()}
    if isinstance(x, pd.DataFrame): return {"columns": [str(c) for c in x.columns], "rows": _py(x.values.tolist())}
    return x
print("__ANSWER__" + json.dumps(_py(ANSWER), ensure_ascii=False, default=str))
"""
RUNNER_PATH = WORK_DIR / "runner.py"
RUNNER_PATH.write_text(RUNNER_SRC, encoding="utf-8")

def run_program(code_str, biz_pkl, cal_pkl, as_of, timeout=EXEC_TIMEOUT):
    """Trả về (answer_dict | None, error_str | None)."""
    with tempfile.NamedTemporaryFile("w", suffix=".py", dir=WORK_DIR, delete=False, encoding="utf-8") as f:
        f.write(code_str); prog = f.name
    try:
        r = subprocess.run([sys.executable, str(RUNNER_PATH), biz_pkl, cal_pkl, str(as_of), prog],
                           capture_output=True, text=True, timeout=timeout, cwd=WORK_DIR)
        out = r.stdout
        if "__ANSWER__" in out:
            ans = json.loads(out.split("__ANSWER__")[-1].strip())
            if isinstance(ans, dict) and "type" in ans:
                return ans, None
            return None, f"ANSWER không đúng định dạng: {str(ans)[:300]}"
        err = (r.stderr or out)[-1500:]
        return None, err or "Không có ANSWER"
    except subprocess.TimeoutExpired:
        return None, f"Timeout sau {timeout}s"
    finally:
        try: os.remove(prog)
        except OSError: pass

def run_many(jobs):
    """jobs: list of (code, biz_pkl, cal_pkl, as_of) -> list of (ans, err), chạy song song."""
    with ThreadPoolExecutor(EXEC_WORKERS) as ex:
        return list(ex.map(lambda j: run_program(*j), jobs))

## 6. LLM backend

- **`vllm`** (dùng khi nộp): load model một lần, sinh **theo lô** cho cả chunk câu hỏi — nhanh hơn nhiều so với gọi tuần tự. `tensor_parallel_size=2` để dùng cả 2 GPU.
- **`openai`**: trỏ tới `vllm serve` chạy local (tiện khi dev trong notebook, có thể restart kernel mà không phải load lại model). **Không** dùng API thương mại cho bài nộp.
- **`mock`**: trả về chương trình cố định — để kiểm tra pipeline khi chưa có GPU.

**Tái lập (±0.03 PA):** cố định `seed` cho từng request (hash của `id` câu hỏi + chỉ số mẫu) → hai lần chạy cho cùng output với cùng phần cứng/phiên bản vLLM.

In [ ]:
def req_seed(qid, k):
    return int(hashlib.md5(f"{qid}-{k}-{SEED}".encode()).hexdigest()[:8], 16)

class LLM:
    def __init__(self, backend=BACKEND, model=MODEL_NAME):
        self.backend, self.model = backend, model
        if self.backend == "vllm":
            try:
                import torch
                from vllm import LLM as _VLLM
                n_gpu = torch.cuda.device_count()
                bf16_ok = n_gpu > 0 and torch.cuda.get_device_capability(0)[0] >= 8   # T4 = 7.5 → fp16
                self.llm = _VLLM(model=model, tensor_parallel_size=int(os.environ.get("TP", max(1, n_gpu))),
                                 dtype="bfloat16" if bf16_ok else "half",
                                 max_model_len=int(os.environ.get("MAX_LEN", 16384)),
                                 gpu_memory_utilization=0.90, seed=SEED, enable_prefix_caching=True)
                self.max_len = self.llm.llm_engine.model_config.max_model_len
                self.tok = self.llm.get_tokenizer()
            except Exception as e:
                print(f"[warn] Không khởi tạo được vLLM ({e}) → chuyển sang backend 'mock'")
                self.backend = "mock"
        elif self.backend == "openai":
            try:
                from openai import OpenAI
                self.client = OpenAI(base_url=OPENAI_BASE, api_key="EMPTY")
            except ImportError:
                self.client = None

    def chat(self, convs, ids, n=1, temperature=0.0, max_tokens=MAX_NEW_TOKENS):
        # trả về list (theo conv) các list n chuỗi
        if self.backend == "mock":
            return [[MOCK_PROGRAM] * n for _ in convs]
        if self.backend == "vllm":
            from vllm import SamplingParams
            # Đo độ dài prompt; prompt quá dài → bỏ qua (trả "" → chương trình lỗi → abstain) thay vì làm crash cả lô
            lens = [len(self.tok.apply_chat_template(c, tokenize=True, add_generation_prompt=True,
                                                     enable_thinking=False)) for c in convs]
            texts_by_conv = [[""] * n for _ in convs]
            keep = [j for j, L in enumerate(lens) if L + 256 <= self.max_len]
            if len(keep) < len(convs):
                print(f"[!] {len(convs) - len(keep)} prompt vượt {self.max_len} token → bỏ qua")
            if not keep:
                return texts_by_conv
            # mỗi mẫu có seed riêng → tái lập được; max_tokens không vượt phần context còn lại
            params = [SamplingParams(n=1, temperature=(0.0 if k == 0 else temperature), top_p=0.95,
                                     max_tokens=min(max_tokens, self.max_len - lens[j]), seed=req_seed(ids[j], k))
                      for j in keep for k in range(n)]
            flat_convs = [convs[j] for j in keep for _ in range(n)]
            res = self.llm.chat(flat_convs, params, use_tqdm=False,
                                chat_template_kwargs={"enable_thinking": False})
            texts = [r.outputs[0].text for r in res]
            for m, j in enumerate(keep):
                texts_by_conv[j] = texts[m * n:(m + 1) * n]
            return texts_by_conv
        if self.backend == "openai":
            def one(args):
                c, i, k = args
                r = self.client.chat.completions.create(model=self.model, messages=c, max_tokens=max_tokens,
                        temperature=(0.0 if k == 0 else temperature), seed=req_seed(i, k),
                        extra_body={"chat_template_kwargs": {"enable_thinking": False}})
                return r.choices[0].message.content
            jobs = [(c, i, k) for c, i in zip(convs, ids) for k in range(n)]
            with ThreadPoolExecutor(32) as ex:
                texts = list(ex.map(one, jobs))
            return [texts[j * n:(j + 1) * n] for j in range(len(convs))]

MOCK_PROGRAM = '''```python
t = next(iter(tables.values()))
ANSWER = {"type": "number", "answer": int(len(t))}
```'''

## 7. Prompt + truy hồi few-shot

Prompt gồm: vai trò & luật trả lời → README business → rulebook → schema profile → **few-shot** → câu hỏi + `as_of_date`.

**Few-shot động:** với mỗi câu test, lấy 3 câu train giống nhất (TF-IDF ký tự n-gram — không tốn tham số model) cùng **chương trình đã giải đúng** của chúng (thu được khi chạy agent trên train, mục 9). Đây là cách rẻ nhất để "dạy" model các mẫu nghiệp vụ lặp lại (top-k sản phẩm, so cùng kỳ, công nợ quá hạn...).

In [ ]:
SYSTEM_PROMPT = """Bạn là chuyên gia phân tích dữ liệu cho hộ kinh doanh nhỏ ở Việt Nam. Viết MỘT chương trình Python trả lời câu hỏi dựa trên dữ liệu.

Môi trường đã có sẵn:
- `tables`: dict tên_bảng -> pandas.DataFrame (đã làm sạch kiểu số/ngày). `sql(query)` chạy DuckDB SQL trên các bảng này (tên bảng như trong schema; bảng `calendar` là lịch âm-dương nếu có).
- `AS_OF`: pd.Timestamp là "hôm nay". Mọi mốc thời gian tương đối (hôm qua, tuần rồi, tháng này, quý trước, cùng kỳ năm ngoái, Tết...) tính theo AS_OF và quy ước trong RULEBOOK.
- `norm(s)`, `find_name(text, series)` (fuzzy match tên trong dữ liệu), pandas `pd`, numpy `np`.

Yêu cầu:
1. Tuân thủ RULEBOOK (đơn hợp lệ, tiêu chí xếp hạng mặc định, cách tính giá vốn/lãi, công nợ quá hạn...). Khi xếp hạng: giảm dần theo tiêu chí, hoà thì theo MÃ tăng dần.
2. Tên sản phẩm/khách hàng có thể viết không thống nhất: gộp các biến thể về cùng một thực thể (dựa vào mã nếu có).
3. Với đáp án dạng tên, trả về NGUYÊN VĂN chuỗi tên lấy từ dữ liệu (không tự viết lại).
4. Cuối chương trình gán: ANSWER = {"type": <type>, "answer": <value>}
   type ∈ number | percent | string | date | boolean | list | table | unanswerable
   - number: số (làm tròn 2 chữ số; tiền VND, số lượng là số nguyên). percent: số phần trăm, vd -12.45 (không có ký hiệu %).
   - date: "yyyy-mm-dd"; hỏi về tháng: "yyyy-mm". list: list có thứ tự. table: {"columns": [...], "rows": [[...], ...]}.
   - unanswerable: khi dữ liệu KHÔNG chứa thông tin cần thiết (vd. hỏi lợi nhuận nhưng không có giá vốn, hỏi khoảng thời gian ngoài phạm vi dữ liệu, hỏi thực thể không tồn tại). Khi đó answer = None.
5. Không đọc file, không dùng mạng. Chỉ in ra rất ít.
Trả lời gồm một khối ```python ... ``` duy nhất."""

def build_user_prompt(b, rulebook, question, as_of, shots):
    # RULEBOOK giống nhau cho mọi câu → đặt ĐẦU TIÊN để vLLM prefix caching dùng chung KV, giảm prefill & bộ nhớ
    parts = []
    if rulebook:    parts.append("## RULEBOOK (của split hiện tại)\n" + rulebook[:12000])
    if b["readme"]: parts.append("## README business\n" + b["readme"][:3000])
    parts.append("## SCHEMA\n" + business_profile(b))
    if shots:
        ex = "\n\n".join(f"Câu hỏi: {s['question']} (as_of {s['as_of_date']})\n```python\n{s['program']}\n```" for s in shots)
        parts.append("## VÍ DỤ ĐÃ GIẢI ĐÚNG (business khác, chỉ tham khảo cách làm)\n" + ex)
    parts.append(f"## CÂU HỎI\nHôm nay (AS_OF) = {as_of}\n{question}")
    return "\n\n".join(parts)

def extract_code(text):
    m = re.findall(r"```(?:python)?\s*\n(.*?)```", text, flags=re.S)
    return (m[-1] if m else text).strip()

# --- few-shot retrieval (TF-IDF char n-gram) ---
from sklearn.feature_extraction.text import TfidfVectorizer
class ShotBank:
    def __init__(self, items):  # items: list of dict(question, as_of_date, program, business_id)
        self.items = items
        self.vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), sublinear_tf=True) if items else None
        if items: self.X = self.vec.fit_transform([norm_str(i["question"]) for i in items])
    def query(self, q, biz_id=None, k=3):
        if not self.items: return []
        sims = (self.X @ self.vec.transform([norm_str(q)]).T).toarray().ravel()
        order = np.argsort(-sims)
        out = [self.items[i] for i in order if self.items[i]["business_id"] != biz_id][:k]  # tránh rò rỉ cùng business
        return out

SHOTS = ShotBank([])

## 8. Agent: sinh → chạy → sửa → bỏ phiếu

Với mỗi chunk câu hỏi:
1. **Sinh K chương trình** (1 greedy + K−1 sampling) theo lô.
2. **Thực thi song song** trong sandbox.
3. **Repair**: chương trình lỗi được gửi lại kèm traceback để sửa (1 vòng).
4. **Chuẩn hoá đáp án** (làm tròn số, format ngày, chuẩn hoá chuỗi) → **bỏ phiếu**. `confidence = số phiếu của đáp án thắng / K`.
5. **Quyết định**: `confidence ≥ τ[type]` → trả lời, ngược lại → `abstain`.

Vì sao self-consistency hoạt động tốt ở đây: các chương trình *đúng* có xu hướng hội tụ về cùng một con số, còn chương trình sai thì sai theo nhiều cách khác nhau → tỉ lệ đồng thuận là tín hiệu calibration rất mạnh và gần như miễn phí.

In [ ]:
VALID_TYPES = {"number", "percent", "string", "date", "boolean", "list", "table", "unanswerable"}

def canonicalize(ans):
    # đưa đáp án về dạng nộp + khoá bỏ phiếu
    t, a = ans.get("type"), ans.get("answer")
    if t not in VALID_TYPES: return None
    try:
        if t == "unanswerable": a = None
        elif t == "number":
            f = float(a); a = int(round(f)) if abs(f - round(f)) < 1e-9 else round(f, 2)
        elif t == "percent": a = round(float(str(a).replace("%", "")), 2)
        elif t == "boolean": a = a if isinstance(a, bool) else str(a).strip().lower() in ("true", "1", "có", "yes")
        elif t == "date":
            s = str(a).strip()
            a = s if re.fullmatch(r"\d{4}-\d{2}(-\d{2})?", s) else pd.Timestamp(s).strftime("%Y-%m-%d")
        elif t == "string":
            if isinstance(a, list) and len(a) == 1: a = a[0]
            a = str(a).strip()
            if not a: return None
        elif t == "list":
            if not isinstance(a, list): return None
            a = [(round(x, 2) if isinstance(x, float) else x) for x in a]
        elif t == "table":
            if not (isinstance(a, dict) and "columns" in a and "rows" in a): return None
    except Exception:
        return None
    return {"type": t, "answer": a}

def _norm_item(x):
    if isinstance(x, str): return norm_str(x)
    if isinstance(x, (list, tuple)): return tuple(_norm_item(y) for y in x)
    if isinstance(x, dict): return json.dumps(x, ensure_ascii=False, default=str, sort_keys=True)
    try: return round(float(x), 2)
    except (TypeError, ValueError): return str(x)

def vote_key(c):
    t, a = c.get("type"), c.get("answer")
    try:
        if t == "string": return (t, norm_str(a))
        if t == "list":   return (t, tuple(_norm_item(x) for x in (a or [])))
        if t == "table":  return (t, json.dumps(sorted(json.dumps(r, ensure_ascii=False, default=str) for r in a["rows"])))
        return (t, json.dumps(a, ensure_ascii=False, default=str))
    except Exception:
        return (t, repr(a))

def solve_chunk(llm, rows, info, rulebook, cal_pkl, n=N_SAMPLES, shots_bank=None):
    shots_bank = shots_bank or SHOTS
    convs, metas = [], []
    for r in rows:
        b = get_business(info, r["business_id"])
        shots = shots_bank.query(r["question"], r["business_id"])
        convs.append([{"role": "system", "content": SYSTEM_PROMPT},
                      {"role": "user", "content": build_user_prompt(b, rulebook, r["question"], r["as_of_date"], shots)}])
        metas.append((r, b))

    gens = llm.chat(convs, [r["id"] for r in rows], n=n, temperature=TEMPERATURE)
    progs = [[extract_code(g) for g in gl] for gl in gens]
    jobs = [(p, b["pkl"], cal_pkl, r["as_of_date"]) for (r, b), pl in zip(metas, progs) for p in pl]
    results = run_many(jobs)
    res = [results[i * n:(i + 1) * n] for i in range(len(rows))]

    # ---- repair round ----
    for _ in range(REPAIR_ROUNDS):
        fix_idx = [(i, k) for i in range(len(rows)) for k in range(n) if res[i][k][0] is None]
        if not fix_idx: break
        fix_convs = [convs[i] + [{"role": "assistant", "content": f"```python\n{progs[i][k]}\n```"},
                                 {"role": "user", "content": f"Chương trình lỗi:\n{res[i][k][1]}\nHãy sửa và trả lại toàn bộ chương trình."}]
                     for i, k in fix_idx]
        fixed = llm.chat(fix_convs, [f"{rows[i]['id']}-fix{k}" for i, k in fix_idx], n=1, temperature=0.0)
        new_progs = [extract_code(f[0]) for f in fixed]
        new_res = run_many([(p, metas[i][1]["pkl"], cal_pkl, rows[i]["as_of_date"]) for p, (i, k) in zip(new_progs, fix_idx)])
        for p, rr, (i, k) in zip(new_progs, new_res, fix_idx):
            progs[i][k], res[i][k] = p, rr

    # ---- vote ----
    outputs = []
    for i, r in enumerate(rows):
        cands = [canonicalize(a) for a, e in res[i] if a is not None]
        cands = [c for c in cands if c is not None]
        if not cands:
            outputs.append({"id": r["id"], "pred": {"type": "abstain", "answer": None}, "conf": 0.0, "programs": progs[i], "cands": []})
            continue
        cnt = Counter(vote_key(c) for c in cands)
        best_key, votes = cnt.most_common(1)[0]
        best = next(c for c in cands if vote_key(c) == best_key)
        win_prog = next(p for p, (a, e) in zip(progs[i], res[i]) if a is not None and canonicalize(a) and vote_key(canonicalize(a)) == best_key)
        outputs.append({"id": r["id"], "pred": best, "conf": votes / n, "programs": progs[i],
                        "win_program": win_prog, "cands": cands})
    return outputs

def decide(out, taus):
    t = out["pred"]["type"]
    tau = taus.get(t, taus.get("_default", DEFAULT_TAU))
    return out["pred"] if (t != "abstain" and out["conf"] >= tau) else {"type": "abstain", "answer": None}

## 9. Chạy trên train & hiệu chỉnh ngưỡng abstain

1. Chạy agent trên (một phần) train → có `conf` và đúng/sai của từng câu.
2. Với từng `type` dự đoán, quét `τ` để **max PA** (chỉ cần ước lượng `P(đúng | conf, type)` > 0.5).
3. Lưu các chương trình **đã đúng** vào `ShotBank` làm few-shot cho test.

> Đánh giá trên các business **không** dùng làm few-shot (đã loại cùng `business_id` trong `ShotBank.query`) để tránh ước lượng lạc quan.

In [ ]:
def run_split(llm, df, info, rulebook, cal_pkl, max_q=None, shots_bank=None, budget_s=None):
    rows = df.to_dict("records")[:max_q] if max_q else df.to_dict("records")
    outs, t_begin = [], time.time()
    for s in range(0, len(rows), CHUNK_SIZE):
        if budget_s and time.time() - t_begin > budget_s:
            print(f"  Hết ngân sách hiệu chỉnh ({budget_s/3600:.1f}h) → dừng ở {s}/{len(rows)} câu"); break
        t0 = time.time()
        outs += solve_chunk(llm, rows[s:s + CHUNK_SIZE], info, rulebook, cal_pkl, shots_bank=shots_bank)
        print(f"  {min(s + CHUNK_SIZE, len(rows))}/{len(rows)} câu, {time.time() - t0:.0f}s/chunk")
    return outs

def calibrate(outs, gold_df, grid=np.round(np.arange(0.2, 1.01, 0.1), 2)):
    g = gold_df.set_index("id")
    recs = []
    for o in outs:
        r = g.loc[o["id"]]
        p = o["pred"]
        recs.append({"type": p["type"], "conf": o["conf"],
                     "s": score_one(p["type"], p["answer"], r.gold_type, r.gold)})
    cal = pd.DataFrame(recs)
    taus = {}
    for t, sub in cal.groupby("type"):
        best = max(grid, key=lambda tau: ((sub.conf >= tau) * sub.s).sum())
        # nếu cả loại này luôn âm → không bao giờ trả lời (τ > 1)
        taus[t] = float(best) if ((sub.conf >= best) * sub.s).sum() > 0 else 1.01
    glob = max(grid, key=lambda tau: ((cal.conf >= tau) * cal.s).sum())
    taus["_default"] = float(glob)
    print("Độ chính xác theo bin confidence:")
    cal["bin"] = pd.cut(cal.conf, [0, .4, .6, .8, 1.0], include_lowest=True)
    print(cal[cal.type != "abstain"].groupby("bin", observed=True).s.agg(
        n="size", precision=lambda x: (x == 1).sum() / max(1, (x != 0).sum())).round(3))
    return taus, cal

llm = None
if not SKIP_CALIB and len(train_df) and "gold_type" in train_df:
    llm = LLM()
    N_CALIB = int(os.environ.get("N_CALIB", 120))
    calib_df = train_df.sample(min(N_CALIB, len(train_df)), random_state=SEED)
    train_outs = run_split(llm, calib_df, train_info, TRAIN_RULEBOOK, TRAIN_CAL_PKL, budget_s=CALIB_BUDGET_S)
    calib_df = calib_df[calib_df["id"].isin({o["id"] for o in train_outs})]

    raw = {o["id"]: (o["pred"]["type"], o["pred"]["answer"]) for o in train_outs}
    print("\nPA nếu trả lời tất cả :", penalized_accuracy(raw, calib_df))
    TAUS, cal_df = calibrate(train_outs, calib_df)
    print("Ngưỡng:", TAUS)
    decided = {o["id"]: tuple(decide(o, TAUS).values()) for o in train_outs}
    print("PA sau abstain (in-sample, lạc quan):", penalized_accuracy(decided, calib_df))

    # ShotBank từ các chương trình đúng
    g = calib_df.set_index("id")
    good = [{"question": g.loc[o["id"], "question"], "as_of_date": g.loc[o["id"], "as_of_date"],
             "business_id": g.loc[o["id"], "business_id"], "program": o["win_program"]}
            for o in train_outs if o.get("win_program") and
               score_one(o["pred"]["type"], o["pred"]["answer"], g.loc[o["id"], "gold_type"], g.loc[o["id"], "gold"]) == 1]
    SHOTS = ShotBank(good)
    ARTIFACTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    json.dump({"taus": TAUS, "shots": good, "n_calib": len(calib_df), "model": MODEL_NAME},
              open(ARTIFACTS_PATH, "w"), ensure_ascii=False, default=str)
    print("Đã lưu", ARTIFACTS_PATH, "→ tải về từ Output và đóng gói vào Docker")
    print(f"ShotBank: {len(good)} chương trình đúng")
else:
    TAUS = {"_default": DEFAULT_TAU}
    art = next((p for p in [ARTIFACTS_PATH, Path("./artifacts.json"), WORK_DIR / "artifacts.json"] if p.exists()), None)
    if art:
        a = json.load(open(art)); TAUS, SHOTS = a["taus"], ShotBank(a["shots"])
        print(f"Dùng artifacts {art}: {len(a['shots'])} shots, ngưỡng {TAUS}")
    else:
        print("[!] Không có artifacts.json → dùng ngưỡng mặc định", DEFAULT_TAU, "và không có few-shot")

**Phân tích lỗi** (làm sau mỗi lần thay đổi): xem các câu sai có `conf` cao — đó là lỗi *hệ thống* (hiểu sai rulebook, sai định nghĩa tuần, quên lọc đơn huỷ...) và thường sửa được hàng loạt.

In [ ]:
if not SKIP_CALIB and len(train_df) and "gold_type" in train_df:
    g = calib_df.set_index("id")
    wrong = [o for o in train_outs if o["conf"] >= 0.8 and
             score_one(o["pred"]["type"], o["pred"]["answer"], g.loc[o["id"], "gold_type"], g.loc[o["id"], "gold"]) == -1]
    print(f"{len(wrong)} câu sai dù confidence ≥ 0.8")
    for o in wrong[:5]:
        r = g.loc[o["id"]]
        print("=" * 100)
        print("Q:", r.question, "| as_of:", r.as_of_date)
        print("GOLD:", r.gold_type, r.gold, "| PRED:", o["pred"], "conf", o["conf"])
        print(o.get("win_program", "")[:1200])

## 10. Chạy test & ghi `submission.jsonl`

Thiết kế an toàn cho luật "quá 6 giờ thì chỉ chấm những dòng đã ghi":
- Ghi **append + flush sau mỗi chunk**; câu chưa kịp xử lý khi hết `TIME_BUDGET_S` được ghi `abstain` (0 điểm, không bị trừ).
- Log từng câu (id, đáp án cuối, thời gian, chương trình thắng, các ứng viên, confidence) ra `/result/logs/logs.jsonl` — BTC **bắt buộc** có log này để đối chiếu khi điểm re-run lệch > 0.03.
- Format theo đề: `abstain` → `{"id", "type": "abstain"}`; `unanswerable` không cần `answer`.

In [ ]:
def write_submission(df, info, path=RESULT_PATH):
    if not len(df):
        raise RuntimeError(f"Test set rỗng — kiểm tra TEST_ROOT={TEST_ROOT}")
    global llm
    llm = llm or LLM()
    rulebook, _, cal_pkl = load_shared(info)
    path.parent.mkdir(parents=True, exist_ok=True)
    done = set()
    if path.exists():  # cho phép chạy tiếp nếu bị ngắt
        done = {json.loads(l)["id"] for l in open(path, encoding="utf-8") if l.strip()}
    rows = [r for r in df.to_dict("records") if r["id"] not in done]
    # nhóm theo business để tận dụng cache + prefix caching
    rows.sort(key=lambda r: r["business_id"])
    LOG_DIR.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as fout, open(LOG_DIR / "logs.jsonl", "a", encoding="utf-8") as flog:
        for s in range(0, len(rows), CHUNK_SIZE):
            chunk = rows[s:s + CHUNK_SIZE]
            if time.time() - T_START > TIME_BUDGET_S:
                for r in rows[s:]:
                    fout.write(json.dumps({"id": r["id"], "type": "abstain"}, ensure_ascii=False) + "\n")
                print("Hết ngân sách thời gian → abstain phần còn lại"); break
            t_chunk = time.time()
            try:
                outs = solve_chunk(llm, chunk, info, rulebook, cal_pkl)
            except Exception as e:
                import traceback; traceback.print_exc()
                print(f"[!] chunk {s} lỗi: {type(e).__name__}: {e} → abstain chunk này")
                outs = [{"id": r["id"], "pred": {"type": "abstain", "answer": None},
                         "conf": 0.0, "programs": [], "cands": []} for r in chunk]
            per_q = (time.time() - t_chunk) / max(1, len(chunk))
            for o in outs:
                p = decide(o, TAUS)
                rec = {"id": o["id"], "type": p["type"]}
                if p["type"] not in ("abstain", "unanswerable"):
                    rec["answer"] = p["answer"]
                fout.write(json.dumps(rec, ensure_ascii=False, default=str) + "\n")
                log = {k: o.get(k) for k in ("id", "pred", "conf", "cands", "win_program")}
                log.update(final=rec, elapsed_s=round(per_q, 2))
                flog.write(json.dumps(log, ensure_ascii=False, default=str) + "\n")
            fout.flush(); flog.flush()
            print(f"{s + len(chunk)}/{len(rows)}  elapsed {(time.time() - T_START) / 60:.1f}'")
    sub = pd.read_json(path, lines=True)
    print(sub.type.value_counts(normalize=True).round(3))

write_submission(test_df, test_info)

## 11. Lộ trình để đạt điểm cao

Xếp theo **tác động / công sức** (làm từ trên xuống). Mỗi bước đều đo bằng PA local trên tập train held-out theo business.

### A. Calibration — đòn bẩy lớn nhất của metric này
1. **Tăng K** (5 → 8–10) chỉ khi đo thấy đường cong precision–coverage tốt hơn; mỗi mẫu thêm tốn thời gian.
2. **Feature cho confidence** ngoài tỉ lệ phiếu: số chương trình lỗi, có phải repair không, `type`, đáp án rỗng/0/NaN, số dòng sau khi lọc = 0, fuzzy-match tên thấp. Huấn luyện một **logistic regression nhỏ** (vài chục tham số, không ảnh hưởng giới hạn 15B) dự đoán `P(đúng)`; trả lời khi > 0.5.
3. **Unanswerable riêng biệt:** chỉ trả `unanswerable` khi đa số phiếu đồng ý *và* có lý do kiểm chứng được (thiếu cột, ngày ngoài phạm vi dữ liệu, thực thể không tồn tại). Đoán `unanswerable` sai cũng bị −1.

### B. Biến rulebook & thời gian thành **công cụ deterministic**
LLM 14B hay sai ở chi tiết nghiệp vụ, nên đừng để nó tự suy diễn mỗi lần:
- `valid_orders(tables)`: lọc đơn hợp lệ theo rulebook (huỷ, trả hàng, đơn nháp...).
- `resolve_period("tuần rồi" | "tháng này" | "cùng kỳ năm ngoái" | "Tết", AS_OF, calendar)` → `(start, end)`, dùng **bảng lịch âm–dương** cho các mốc âm lịch.
- `rank(df, by, k)` với tie-break **mã tăng dần** đúng như luật chấm.
- `cogs()`, `gross_profit()`, `overdue_debt()` theo đúng công thức rulebook.

Đưa các hàm này vào runner và mô tả trong prompt → model chỉ còn phải *chọn và ghép* công cụ. Đây là logic chung (không gắn với id câu hỏi/business) nên hợp lệ theo luật 4.

### C. Chuẩn hoá dữ liệu & thực thể theo business
- **Entity resolution:** gom các biến thể tên ("Sting dâu 330ml", "sting dau lon", mã `SP0042`) về một mã chuẩn bằng rapidfuzz + mã sản phẩm, tạo sẵn bảng `entities(code, canonical_name, aliases)`. Vì bộ chấm chấp nhận **mã** và mọi cách viết trong dữ liệu, trả về tên nguyên văn từ dữ liệu là an toàn.
- Gộp bảng nhiều sheet cùng cấu trúc (vd. mỗi tháng một sheet) thành một bảng dài có cột `thang`.
- Kiểm tra lại loader trên toàn bộ train: in các bảng có header lạ, cột số còn ở dạng text, dòng tổng sót.

### D. Fine-tune model ≤15B bằng dữ liệu tự sinh (được phép theo luật 6)
1. **Rejection sampling trên train:** sinh nhiều chương trình/câu, giữ lại chương trình cho đáp án đúng → cặp (prompt, program) chất lượng cao.
2. **Sinh câu hỏi tổng hợp:** viết template câu hỏi tiếng Việt (top-k, tổng theo kỳ, so sánh %, công nợ...) + biến thể văn nói/viết tắt ("sp", "dc", "bn", "tuần rồi"), tính đáp án bằng code chuẩn → hàng chục nghìn mẫu. Thêm cả câu **unanswerable** có chủ đích.
3. **LoRA SFT** Qwen-14B trên các trace này (2×5090 đủ cho QLoRA). Thường là bước nhảy lớn nhất về accuracy sau khi pipeline đã ổn.

### E. Tốc độ (cũng là tie-breaker)
- Ước lượng thời gian/câu trên public test × số câu private dự kiến; giữ biên ≥ 20%.
- FP8/AWQ để tăng throughput; `enable_prefix_caching` (đã bật) + sắp câu theo business để tái dùng prefix README/schema/rulebook.
- Early-stop voting: nếu 3 mẫu đầu đã trùng nhau thì bỏ qua các mẫu còn lại.
- Có thể bật `enable_thinking=True` cho Qwen3 ở riêng các câu confidence thấp (vòng 2), thay vì cho mọi câu.

### F. Tái lập & đóng gói
- Seed theo request (đã làm), cố định phiên bản vLLM/PyTorch trong Docker, chạy 2 lần trên public test và kiểm tra |ΔPA| ≪ 0.03.
- `README` khai báo model & tổng tham số; không truy cập mạng (chạy thử với `--network none`).
- Đọc `/data` read-only, ghi tạm vào `/tmp`, kết quả `/result/submission.jsonl`.

### G. Tổng quát hoá sang business mới
Private test có **loại hình kinh doanh chưa từng xuất hiện** và README chất lượng không đều (có cái gần như trống). Đừng dựa vào tên cột cố định; để agent tự đọc schema profile, và đánh giá local theo kiểu **leave-business-type-out** (giữ lại cả một loại hình như nhà thuốc làm validation) để ước lượng đúng điểm private.

### H. Kaggle vs Docker nộp bài
- Kaggle (2×T4, AWQ) dùng để dev, hiệu chỉnh ngưỡng, chạy public test.
- Điểm cuối = BTC chạy lại Docker trên 2×RTX 5090 và phải khớp file bạn nộp trong ±0.03. Nếu Docker dùng bf16 mà bạn chạy private test trên Kaggle bằng AWQ thì đáp án có thể lệch. **Dùng cùng một bản weights (cùng quantization) và cùng cấu hình** ở cả hai nơi; tốt nhất thuê 5090 theo giờ để chạy private test bằng đúng image đã nộp.
- Ngưỡng τ hiệu chỉnh với model nào thì chỉ đúng cho model đó — đổi model/quantization là phải hiệu chỉnh lại.

### Checklist trước khi nộp
- [ ] Scorer local khớp điểm public leaderboard (sai lệch nhỏ).
- [ ] Mọi dòng trong test đều có trong submission (thiếu = abstain, nhưng nên ghi rõ).
- [ ] Không có type lạ, `percent` không kèm `%`, `date` đúng định dạng.
- [ ] Thời gian chạy toàn bộ < 6h kể cả load model, với biên an toàn.